In [2]:
!pip install pyodbc

  Using cached pyodbc-5.2.0-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
Using cached pyodbc-5.2.0-cp313-cp313-win_amd64.whl (69 kB)


In [3]:
import pyodbc

# Connection details for source and destination databases
source_server = 'HYPE\SQL_2022'
source_database = 'CRM'
source_uid = 'sa'
source_pwd = 'D@1ry$2014'

dest_server = 'HYPE\SQL_2022'
dest_database = 'OLTP'
dest_uid = 'sa'
dest_pwd = 'D@1ry$2014'

source_table_name = 'CrmCek'
dest_table_name = 'CrmCek'

try:
    # Connect to source database
    source_cnxn = pyodbc.connect(f'DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={source_server};DATABASE={source_database};UID={source_uid};PWD={source_pwd}')
    source_cursor = source_cnxn.cursor()

    # Connect to destination database
    dest_cnxn = pyodbc.connect(f'DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={dest_server};DATABASE={dest_database};UID={dest_uid};PWD={dest_pwd}')
    dest_cursor = dest_cnxn.cursor()

    # Get schema and data from source table
    source_cursor.execute(f"SELECT * FROM {source_table_name}")
    columns = [column[0] for column in source_cursor.description]
    rows = source_cursor.fetchall()

    # Construct CREATE TABLE statement for destination
    column_definitions = []
    for i, col_name in enumerate(columns):
        # You might need more sophisticated type mapping here
        # This is a basic example and might not cover all SQL Server data types
        sql_type = source_cursor.description[i][1] # Get ODBC type code
        if sql_type == pyodbc.SQL_VARCHAR:
            column_definitions.append(f"{col_name} VARCHAR({source_cursor.description[i][3]})")
        elif sql_type == pyodbc.SQL_INTEGER:
            column_definitions.append(f"{col_name} INT")
        # Add more type mappings as needed
        else:
            column_definitions.append(f"{col_name} NVARCHAR(MAX)") # Fallback for unknown types

    create_table_sql = f"CREATE TABLE {dest_table_name} ({', '.join(column_definitions)})"
    dest_cursor.execute(create_table_sql)
    dest_cnxn.commit()

    # Insert data into destination table
    if rows:
        placeholders = ', '.join(['?' for _ in columns])
        insert_sql = f"INSERT INTO {dest_table_name} VALUES ({placeholders})"
        dest_cursor.executemany(insert_sql, rows)
        dest_cnxn.commit()

    print(f"Table '{source_table_name}' copied to '{dest_table_name}' successfully.")

except pyodbc.Error as ex:
    sqlstate = ex.args[0]
    print(f"Database error: {sqlstate} - {ex}")
finally:
    if 'source_cnxn' in locals() and source_cnxn:
        source_cnxn.close()
    if 'dest_cnxn' in locals() and dest_cnxn:
        dest_cnxn.close()

<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\S'
C:\Users\HYPE AMD\AppData\Local\Temp\ipykernel_6856\1716270394.py:4: SyntaxWarning: invalid escape sequence '\S'
  source_server = 'HYPE\SQL_2022'
C:\Users\HYPE AMD\AppData\Local\Temp\ipykernel_6856\1716270394.py:9: SyntaxWarning: invalid escape sequence '\S'
  dest_server = 'HYPE\SQL_2022'


Table 'CrmCek' copied to 'CrmCek' successfully.
